# academy-cairn — live demo

Trust-aware **Academy** agents backed by a live **Cairn** service. Each demo
launches real Academy agents and reads/writes real reputation scores.

**Run this notebook from the `examples/` directory** (so `import _demo` resolves),
**one cell at a time** — each demo's agent flushes its ratings on shutdown, so give
it a moment before reading the score in the next demo.

The only trust-specific code is the `CairnAgentMixin` base and a `@cairn_guarded`
(or `rated(...)`) line — everything else is a normal Academy agent.

In [ ]:
import os

from _demo import academy, bar, demo_entity, reader
from academy.agent import Agent, action
from academy.handle import Handle

from academy_cairn import (
    CairnAgentMixin,
    CairnTrustError,
    EntityRef,
    TrustPolicy,
    cairn_guarded,
    rated,
)

# Group these agents' identities; point CAIRN_BASE_URL at your own Cairn to switch.
os.environ.setdefault("CAIRN_NAMESPACE", "demo")
# os.environ["CAIRN_BASE_URL"] = "http://localhost:8000"

read = reader()   # a read-only client for looking up scores

## 1. Trust accumulates

A source starts with no track record (`0.5 / confidence 0.0` = *no data*, not *bad*).
Eight good fetches earn it a reputation.

In [ ]:
WEATHER_API = demo_entity("weather-api")

class WeatherAgent(CairnAgentMixin, Agent):
    @action
    @cairn_guarded(type="data_source", id_from="url")
    async def fetch(self, url: str) -> str:
        return "72F, sunny"

async def demo1():
    ref = EntityRef(type="data_source", external_id=WEATHER_API)
    before = await read.get_score(ref)
    print(f"before:  trust {bar(before.composite_score)} {before.composite_score:.2f}"
          f"   evidence {before.confidence:.2f}")
    async with academy() as manager:
        agent = await manager.launch(WeatherAgent, name="weather-agent")
        for _ in range(8):
            await agent.fetch(url=WEATHER_API)
    after = await read.get_score(ref)
    print(f"after:   trust {bar(after.composite_score)} {after.composite_score:.2f}"
          f"   evidence {after.confidence:.2f}")

await demo1()

## 2. Reputations diverge, and the guard acts

A reliable and a flaky source pull apart; a strict agent then blocks the one it
can't trust — but never blocks a brand-new source (no evidence ⇒ always allowed).

In [ ]:
RELIABLE, FLAKY, UNKNOWN = (demo_entity("reliable-api"),
                            demo_entity("flaky-api"),
                            demo_entity("brand-new-api"))
STRICT = TrustPolicy(min_score=0.5, min_confidence=0.3, on_low="block")

class SeederAgent(CairnAgentMixin, Agent):
    @action
    @cairn_guarded(type="data_source", id_from="url")
    async def fetch(self, url: str) -> str:
        if url == FLAKY:
            raise TimeoutError("source timed out")
        return "ok"

class ResearchAgent(CairnAgentMixin, Agent):
    @action
    @cairn_guarded(type="data_source", id_from="url", policy=STRICT)
    async def query(self, url: str) -> str:
        return f"<results from {url}>"

async def demo2():
    async with academy() as manager:
        seeder = await manager.launch(SeederAgent, name="seeder")
        for _ in range(8):
            await seeder.fetch(url=RELIABLE)
            try:
                await seeder.fetch(url=FLAKY)
            except TimeoutError:
                pass
    for label, url in [("reliable", RELIABLE), ("flaky", FLAKY)]:
        r = await read.get_score(EntityRef(type="data_source", external_id=url))
        print(f"{label:9s} trust {bar(r.composite_score)} {r.composite_score:.2f}")
    print()
    async with academy() as manager:
        agent = await manager.launch(ResearchAgent, name="research-agent")
        sources = [("reliable ", RELIABLE), ("flaky    ", FLAKY),
                   ("brand-new", UNKNOWN)]
        for label, url in sources:
            try:
                await agent.query(url=url)
                print(f"{label}  allowed")
            except CairnTrustError:
                print(f"{label}  BLOCKED (trust too low)")

await demo2()

## 3. Score your peer agents

`rated(handle)` scores a peer from what actually happened when you call it —
answered, without error, in time — not from analyzing its output.

In [ ]:
import uuid

run = uuid.uuid4().hex[:6]
analyzer_name, worker_name = f"analyzer-{run}", f"flaky-worker-{run}"

class Analyzer(Agent):
    @action
    async def analyze(self, data: str) -> str:
        return f"analysis of {data}"

class FlakyWorker(Agent):
    def __init__(self) -> None:
        super().__init__()
        self._calls = 0
    @action
    async def analyze(self, data: str) -> str:
        self._calls += 1
        if self._calls % 3 != 0:
            raise RuntimeError("worker crashed")
        return f"analysis of {data}"

class Coordinator(CairnAgentMixin, Agent):
    def __init__(self, analyzer: Handle, worker: Handle) -> None:
        super().__init__()
        self._analyzer, self._worker = analyzer, worker
    @action
    async def delegate(self, rounds: int) -> None:
        analyzer = rated(self._analyzer, cairn=self.cairn)
        worker = rated(self._worker, cairn=self.cairn)
        for _ in range(rounds):
            await analyzer.analyze("dataset")
            try:
                await worker.analyze("dataset")
            except RuntimeError:
                pass

async def demo3():
    async with academy() as manager:
        analyzer = await manager.launch(Analyzer, name=analyzer_name)
        worker = await manager.launch(FlakyWorker, name=worker_name)
        coordinator = await manager.launch(Coordinator, args=(analyzer, worker))
        await coordinator.delegate(8)
    ns = read.config.namespace
    for label, name in [("analyzer", analyzer_name), ("flaky-worker", worker_name)]:
        ref = EntityRef(type="agent", external_id=f"agent://academy/{ns}/{name}")
        r = await read.get_score(ref)
        print(f"{label:13s} {bar(r.composite_score)} {r.composite_score:.2f}")

await demo3()